# NYC TLC Yellow Taxi — Predictive Analysis

Notebook này tái dựng câu chuyện phân tích từ các output do pipeline đã kiểm tra tạo ra. Mục tiêu là trình bày gọn, truy vết được số liệu và không âm thầm fit lại mô hình sau khi final test đã mở. Quy trình rebuild đầy đủ từ raw data nằm trong `scripts/run_pipeline.ps1`.

## 1. Thiết lập và kiểm tra artifact

Notebook có thể chạy từ workspace root hoặc trực tiếp trong thư mục `notebooks`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

candidates = [Path.cwd() / "nyc_tlc_project", Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "outputs").exists() and (p / "config").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy nyc_tlc_project. Hãy chạy notebook từ workspace root hoặc thư mục notebooks.")
OUTPUTS = PROJECT_ROOT / "outputs"

def load_json(relative_path):
    return json.loads((PROJECT_ROOT / relative_path).read_text(encoding="utf-8"))

required = [
    OUTPUTS / "audit/audit_summary.json",
    OUTPUTS / "eda/eda_summary.json",
    OUTPUTS / "modeling/mean_fold_metrics.csv",
    OUTPUTS / "final_evaluation/final_metrics.csv",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Thiếu artifact: {missing}"
print(f"Project root: {PROJECT_ROOT}")
print("Artifact gate: PASS")

## 2. Bài toán và thiết kế đánh giá

- **Stakeholder giả định:** nhóm vận hành Yellow Taxi hoặc đơn vị hỗ trợ tài xế.
- **Observation:** một pickup zone × block 4 giờ.
- **Target:** số Yellow Taxi pickup đã ghi nhận trong block kế tiếp.
- **Primary metric:** MAE.
- **Primary baseline:** cùng zone-block một tuần trước.
- **Development:** 01/2024–12/2025; bốn rolling folds trong 2025.
- **Final untouched test:** 01/2026–05/2026.

Target là realized pickup volume, không phải latent passenger demand.

In [ ]:
audit = load_json("outputs/audit/audit_summary.json")
panel = load_json("outputs/panel/panel_build_summary.json")
final_build = load_json("outputs/final_test_build/final_build_summary.json")

scope = pd.DataFrame([
    {"Tập": "Development raw", "Giai đoạn": "2024-01 đến 2025-12", "Rows": audit["raw_rows"]},
    {"Tập": "Development panel", "Giai đoạn": "2024-01 đến 2025-12", "Rows": panel["panel_rows"]},
    {"Tập": "Final raw", "Giai đoạn": "2026-01 đến 2026-05", "Rows": final_build["raw_rows"]},
    {"Tập": "Final panel", "Giai đoạn": "2026-01 đến 2026-05", "Rows": final_build["panel_rows"]},
])
scope.style.format({"Rows": "{:,.0f}"})

## 3. Data quality và panel

Cleaning chỉ loại record không thể gán đáng tin cậy vào đúng pickup month/zone và exact duplicate. Fare, duration, distance và passenger anomalies được audit nhưng không dùng để thay đổi pickup-count target.

In [ ]:
quality = pd.DataFrame([
    ["Raw development rows", audit["raw_rows"]],
    ["Exact duplicates removed", audit["target_valid_exact_duplicate_rows"]],
    ["Clean pickup target rows", panel["panel_target_sum"]],
    ["Eligible zones", panel["eligible_zones"]],
    ["Development panel rows", panel["panel_rows"]],
    ["Zero-cell share (%)", panel["zero_cell_pct"]],
], columns=["Chỉ tiêu", "Giá trị"])
quality.style.format({"Giá trị": lambda x: f"{x:,.2f}"})

## 4. EDA và problem discovery

EDA cho thấy pattern tuần mạnh, pickup tập trung vào ít zone, độ thưa khác biệt lớn theo activity tier và mức volume thay đổi giữa 2024–2025.

In [ ]:
eda = load_json("outputs/eda/eda_summary.json")
eda_kpis = pd.DataFrame({
    "Chỉ tiêu": ["Top 10 zone share", "Top 20 zone share", "Time-of-week profile corr. 2024–2025", "Previous-week within-zone corr.", "Daily mean change 2025 vs 2024"],
    "Giá trị": [eda["top_10_zone_share_pct"], eda["top_20_zone_share_pct"], eda["time_of_week_profile_correlation_2024_2025"], eda["previous_week_lag"]["pooled_within_zone_centered_correlation"], eda["daily_mean_change_2025_vs_2024_pct"]],
    "Đơn vị": ["%", "%", "correlation", "correlation", "%"],
})
eda_kpis.style.format({"Giá trị": "{:.3f}"})

In [ ]:
lag = pd.read_csv(OUTPUTS / "eda/lag_correlations.csv")
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=lag, x="lag", y="pooled_within_zone_centered_correlation", color="#4C78A8", ax=ax)
ax.set(title="Historical pickup signal after centering within zone", xlabel="Lag", ylabel="Correlation")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 5. Feature set và leakage controls

Feature đã khóa gồm zone identity, calendar fields biết trước, lags 1/6/42/84/168, four-week seasonal mean và shifted rolling statistics. Mọi rolling feature được shift trước khi tính; encoder/scaler chỉ fit trên training portion; validation theo thời gian, không random split.

In [ ]:
feature_summary = load_json("outputs/modeling_spec/feature_build_summary.json")
validation_folds = pd.read_csv(OUTPUTS / "modeling_spec/validation_folds.csv")
print(f"Feature-ready rows: {feature_summary['feature_ready_rows']:,}")
print(f"Validation rows: {feature_summary['validation_rows_total']:,}")
print(f"Leakage audit: {'PASS' if feature_summary['leakage_audit_success'] else 'FAIL'}")
validation_folds

## 6. Rolling validation — development results

Formal selection tuân theo rule đã khóa: regression chỉ thay previous-week nếu cải thiện MAE đủ ổn định và không làm suy giảm nghiêm trọng low-activity tier.

In [ ]:
development = pd.read_csv(OUTPUTS / "modeling/mean_fold_metrics.csv").sort_values("mean_MAE")
cols = ["display_name", "mean_MAE", "mean_RMSE", "mean_WAPE_pct", "mean_R2", "uplift_vs_previous_week_pct"]
development[cols].style.format({c: "{:.3f}" for c in cols[1:]})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_data = development.sort_values("mean_MAE", ascending=False)
colors = ["#F28E2B" if m == "baseline_previous_week" else "#4C78A8" for m in plot_data["model"]]
ax.barh(plot_data["display_name"], plot_data["mean_MAE"], color=colors)
ax.set(title="Mean MAE across four rolling validation folds", xlabel="MAE (pickups per zone-block)", ylabel="")
for i, value in enumerate(plot_data["mean_MAE"]):
    ax.text(value + 0.4, i, f"{value:.2f}", va="center")
plt.tight_layout()
plt.show()

Four-week mean có development MAE thấp nhất (14,88). Ridge giảm aggregate MAE nhưng vi phạm low-tier gate. Do không regression nào đạt toàn bộ replacement rules, previous-week (MAE 16,39) là formal frozen selection.

## 7. Frozen final test 01/2026–05/2026

Final test chỉ được mở sau khi feature set, Ridge alpha và formal selection đã khóa. Kết quả test không được dùng để chọn lại model.

In [ ]:
final_metrics = pd.read_csv(OUTPUTS / "final_evaluation/final_metrics.csv").sort_values("MAE")
final_cols = ["display_name", "MAE", "RMSE", "WAPE_pct", "R2", "mean_bias"]
final_metrics[final_cols].style.format({c: "{:.3f}" for c in final_cols[1:]})

In [ ]:
segments = pd.read_csv(OUTPUTS / "final_evaluation/final_segment_metrics.csv")
monthly = segments[(segments["dimension"] == "month") & segments["method"].isin(["baseline_previous_week", "baseline_seasonal_mean_4w", "ridge_regression"])]
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.lineplot(data=monthly, x="segment", y="MAE", hue="display_name", marker="o", ax=ax)
ax.set(title="Final-test MAE by month", xlabel="Month", ylabel="MAE (pickups per zone-block)")
ax.legend(title="Method", frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
tiers = segments[(segments["dimension"] == "activity_tier") & segments["method"].isin(["baseline_previous_week", "baseline_seasonal_mean_4w", "ridge_regression"])]
tiers.pivot(index="segment", columns="display_name", values="MAE").round(3)

## 8. Kết luận

- Formal previous-week method đạt **MAE 16,22; RMSE 55,74; WAPE 20,22%; R² 0,935** trên final test.
- Ridge có final MAE thấp nhất (**14,17**) nhưng chỉ là empirical result; frozen decision không đổi.
- Four-week mean đạt MAE **14,64** và tốt nhất trong 4/5 tháng.
- Ridge tốt hơn ở high-activity zones nhưng kém ở sparse/zero-demand observations.
- Previous-week dễ truyền một shock sang đúng tuần kế tiếp.

Dự báo phù hợp làm tín hiệu hỗ trợ demand review, chưa đủ để tuyên bố latent demand, thiếu tài xế, lợi nhuận, quan hệ nhân quả hoặc phương án dispatch tối ưu.

In [ ]:
verification_files = [
    "outputs/audit/audit_verification.json",
    "outputs/eda/eda_verification.json",
    "outputs/modeling_spec/modeling_spec_verification.json",
    "outputs/modeling/modeling_verification.json",
    "outputs/final_evaluation/final_evaluation_verification.json",
]
verification = []
for path in verification_files:
    payload = load_json(path)
    verification.append({"artifact": Path(path).stem, "success": payload.get("success"), "errors": len(payload.get("errors", []))})
verification = pd.DataFrame(verification)
assert verification["success"].all() and (verification["errors"] == 0).all()
verification